# Movie Sentiment Analysis
A simple walkthrough of TF-IDF and Logistic Regression for movie-review sentiment.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'ml'))
from preprocess import preprocess_text

## Load and inspect the dataset

In [ ]:
data = pd.read_csv(ROOT / 'data' / 'sample_imdb_reviews.csv')
display(data.head())
print(data.shape)
print(data.isnull().sum())

## Visualize the sentiment distribution

In [ ]:
sns.countplot(data=data, x='sentiment', hue='sentiment', legend=False, palette='Set2')
plt.title('Sentiment distribution')
plt.show()

## Clean reviews and create labels

In [ ]:
data['clean_review'] = data['review'].apply(preprocess_text)
data['label'] = data['sentiment'].map({'positive': 1, 'negative': 0})
display(data[['review', 'clean_review', 'label']].head())

## Train/test split, TF-IDF, and Logistic Regression

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    data['clean_review'], data['label'], test_size=0.25, random_state=42, stratify=data['label']
)
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
x_train_tfidf = vectorizer.fit_transform(x_train)
x_test_tfidf = vectorizer.transform(x_test)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(x_train_tfidf, y_train)
predictions = model.predict(x_test_tfidf)

## Evaluate the model

In [ ]:
print('Accuracy:', round(accuracy_score(y_test, predictions), 4))
print(classification_report(y_test, predictions, target_names=['negative', 'positive']))
matrix = confusion_matrix(y_test, predictions)
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix'); plt.show()

## Test sample reviews

In [ ]:
samples = ['This movie was fantastic. The acting was excellent and I really enjoyed it.', 'This movie was terrible. The story was boring and the acting was disappointing.']
for review in samples:
    cleaned = preprocess_text(review)
    prediction = model.predict(vectorizer.transform([cleaned]))[0]
    print(review, '->', 'Positive' if prediction else 'Negative')